In [ ]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException
import pandas as pd
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.edge.options import Options as EdgeOptions
from selenium.webdriver.edge.service import Service as EdgeService
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
from selenium import webdriver
from webdriver_manager.microsoft import EdgeChromiumDriverManager
import pyautogui as py

from pathlib import Path
from dotenv import load_dotenv
import re
import time
import shutil
import os
import requests

# carrega as variáveis do .env
load_dotenv()

# credenciais
login = os.getenv("LOGIN")
senha = os.getenv("SENHA")

if not login or not senha:
    raise ValueError("LOGIN ou SENHA não encontrados no .env. Verifique o arquivo.")

planilha = "hashs.csv"
hash_pasta = pd.read_csv(planilha)

# pagina de login
login_url = "https://ww4.tce.ms.gov.br/tcedigital-protocolo/login/"

# pagina de pesquisa/trabalho
segundo_url = "https://ww4.tce.ms.gov.br/tcedigital-protocolo/esfinge/remessa/extrato"

# diretório base onde as pastas serão criadas
BASE_CONTRATO = Path(r"C:\\Users\\ml_be\\Downloads\\Contrato")

# diretório de download do Edge
DOWNLOAD_DIR = Path(r"C:\\Users\\ml_be\\Downloads")

def criar_driver_edge(download_dir: Path):
    options = EdgeOptions()
    prefs = {
        "download.default_directory": str(download_dir.resolve()),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
    }
    options.add_experimental_option("prefs", prefs)
    driver = webdriver.Edge(options=options)
    driver.maximize_window()
    return driver

def cookies_selenium_para_requests(driver):
    s = requests.Session()
    for c in driver.get_cookies():
        s.cookies.set(c["name"], c["value"], domain=c.get("domain"), path=c.get("path", "/"))
    return s


def esperar_download_pdf(download_dir: Path, iniciado_em: float, timeout: int = 180) -> Path:
    """
    Espera um PDF baixado APÓS o momento 'iniciado_em' (time.time()).
    Retorna o PDF mais recente que tenha mtime >= iniciado_em e sem download ativo.
    """
    download_dir = Path(download_dir)
    fim = time.time() + timeout

    # Edge (Chromium) pode usar .crdownload
    while time.time() < fim:
        if list(download_dir.glob("*.crdownload")):
            time.sleep(0.5)
            continue

        pdfs = [p for p in download_dir.glob("*.pdf") if p.is_file()]
        pdfs_novos = [p for p in pdfs if p.stat().st_mtime >= iniciado_em]

        if pdfs_novos:
            pdfs_novos.sort(key=lambda p: p.stat().st_mtime, reverse=True)
            return pdfs_novos[0]

        time.sleep(0.5)

    raise TimeoutError(f"PDF não apareceu em {timeout}s (após timestamp) em: {download_dir}")


def sanitizar_nome(texto: str) -> str:
    texto = str(texto).strip()
    texto = re.sub(r'[<>:"/\\\\\\\\|?*]+', "_", texto)
    texto = re.sub(r"\\s+", " ", texto).strip()
    return texto


def criar_driver_edge(download_dir: Path):
    options = EdgeOptions()

    # Preferências de download (Edge/Chromium)
    prefs = {
        "download.default_directory": str(download_dir.resolve()),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
    }
    options.add_experimental_option("prefs", prefs)

    # Selenium Manager (Selenium 4.6+): não usa webdriver_manager
    driver = webdriver.Edge(options=options)
    driver.maximize_window()
    return driver


def esperar_download_xls(download_dir: Path, iniciado_em: float, timeout=180) -> Path:  # ✅ adicionado iniciado_em
    """
    Espera terminar download e retorna o arquivo .xls/.xlsx mais recente.
    """
    inicio = time.time()
    download_dir = Path(download_dir)

    while time.time() - inicio < timeout:
        if list(download_dir.glob("*.crdownload")):
            time.sleep(1)
            continue

        arquivos = []
        arquivos.extend(download_dir.glob("*.xls"))
        arquivos.extend(download_dir.glob("*.xlsx"))

        novos = [p for p in arquivos if p.stat().st_mtime >= iniciado_em]  # ✅ filtra por timestamp

        if novos:
            novos.sort(key=lambda p: p.stat().st_mtime, reverse=True)
            return novos[0]

        time.sleep(1)

    raise TimeoutError(f"Download não apareceu em {timeout}s em {download_dir}")


driver = criar_driver_edge(DOWNLOAD_DIR)
wait = WebDriverWait(driver, 20)

###### Login #####

driver.get(login_url)
wait.until(EC.element_to_be_clickable((By.XPATH, "/html/body/app-root/app-layout-login/app-login/div/div/div/div[1]/div/div[2]/div[2]/form/div/div[1]/input"))).send_keys(login)

wait.until(EC.element_to_be_clickable((By.XPATH, "/html/body/app-root/app-layout-login/app-login/div/div/div/div[1]/div/div[2]/div[2]/form/div/div[2]/input"))).send_keys(senha)

time.sleep(30)  # tempo para marcar reCAPTCHA manualmente

wait.until(EC.element_to_be_clickable((By.XPATH, "/html/body/app-root/app-layout-login/app-login/div/div/div/div[1]/div/div[2]/div[2]/form/div/div[4]/div[1]/button"))).click()

time.sleep(5)

# acessa a pagina de pesquisa/trabalho
driver.get(segundo_url)
time.sleep(5)

for index, row in hash_pasta.iterrows():
    codigo_hash = row["codigo_hash"]

    # cria pasta desta iteração
    BASE_CONTRATO.mkdir(parents=True, exist_ok=True)
    pasta_destino = BASE_CONTRATO / sanitizar_nome(codigo_hash)
    pasta_destino.mkdir(parents=True, exist_ok=True)

    print(f"[{index}] Hash: {codigo_hash} | Pasta: {pasta_destino}")

    # pesquisa
    campo = wait.until(EC.element_to_be_clickable((By.XPATH, '/html/body/app-root/app-layout-padrao/div/div/div/app-spa-host/div/esfinge-root/esfinge-main/esfinge-remessa/tce-panel/div/div/div/div[1]/form/esfinge-filtro-remessa/div/div[5]/tce-input/div/input')))
    campo.clear()
    campo.send_keys(codigo_hash)
    time.sleep(3)
    ActionChains(driver).send_keys(Keys.ENTER).perform()
    time.sleep(3)

    wait.until(EC.element_to_be_clickable((By.XPATH, '/html/body/app-root/app-layout-padrao/div/div/div/app-spa-host/div/esfinge-root/esfinge-main/esfinge-remessa/tce-panel/div/div/div/div[2]/esfinge-extrato/esfinge-remessa-pagina/tce-tab/div/div/tce-collapse/ngb-accordion/div/div[2]/div/div/tce-grid/div/table/tbody/tr/td[4]/div/tce-button/button'))).click()
    time.sleep(2)

    # ESTE clique dispara o download do XLS
    t0_xls = time.time()  # ✅ marco de tempo antes do clique
    wait.until(EC.element_to_be_clickable((By.XPATH, '/html/body/app-root/app-layout-padrao/div/div/div/app-spa-host/div/esfinge-root/esfinge-main/esfinge-remessa/tce-panel/div/div/div/div[2]/esfinge-extrato/esfinge-remessa-pagina/tce-tab/div/div/div/div[3]/div/tce-grid-paginate/tce-grid/div/table/tbody/tr/td[9]/span[2]/i'))).click()

    # espera o download finalizar e move para a pasta destino com nome "dados.ext"
    arquivo_baixado = esperar_download_xls(DOWNLOAD_DIR, iniciado_em=t0_xls, timeout=15)  # ✅ passa t0_xls
    ext = arquivo_baixado.suffix.lower()

    destino_final = pasta_destino / f"dados{ext}"
    if destino_final.exists():
        destino_final.unlink()  # sobrescreve

    shutil.move(str(arquivo_baixado), str(destino_final))
    print(f"[+] Salvo: {destino_final}")

    # abre a guia de dados da remessa
    time.sleep(2)
    wait.until(EC.element_to_be_clickable((By.XPATH, '/html/body/app-root/app-layout-padrao/div/div/div/app-spa-host/div/esfinge-root/esfinge-main/esfinge-remessa/tce-panel/div/div/div/div[2]/esfinge-extrato/esfinge-remessa-pagina/tce-tab/div/div/div/div[3]/div/tce-grid-paginate/tce-grid/div/table/tbody/tr/td[9]/span[1]/i'))).click()
    time.sleep(3)

    # abre a lista de arquivos(SEMPRE ALTERAR ESSE XPATH CONFORME ONDE SE CLICA PRA ACESSAR OS ARQUIVOS PDFS)
    driver.find_element(By.XPATH, '/html/body/app-root/app-layout-padrao/div/div/div/app-spa-host/div/esfinge-root/esfinge-main/esfinge-remessa/tce-panel/div/div/div/div[2]/esfinge-extrato/esfinge-remessa-pagina/tce-tab/div/div/div/div[3]/div[2]/tce-tab/div/div/div/tce-grid-paginate/tce-grid/div/table/tbody/tr/td[17]/tce-button/button').click()
    time.sleep(4)

    # ─────────────────────────────────────────────────────────────
    # BAIXAR TODOS OS PDFs DA TABELA (tr[1..N])
    # ─────────────────────────────────────────────────────────────
    tbody_xpath = "/html/body/app-root/app-layout-padrao/div/div/div/app-spa-host/div/esfinge-root/esfinge-main/esfinge-remessa/tce-panel/div/div/div/div[2]/esfinge-extrato/esfinge-remessa-pagina/tce-tab/div/div/div/div[3]/div[2]/tce-tab/div/div/div/tce-grid/div/table/tbody"
    
    linhas = driver.find_elements(By.XPATH, f"{tbody_xpath}/tr")  # conta linhas
    qtd = len(linhas)

    print(f"[*] Arquivos encontrados na lista: {qtd}. {now}")

    MAX_TENTATIVAS = 3
    ESPERA_ENTRE_TENTATIVAS = 5

    for L in range(1, qtd + 1):
        nome_xpath = f"{tbody_xpath}/tr[{L}]/td[1]/span"
        pdf_xpath  = f"{tbody_xpath}/tr[{L}]/td[3]/tce-button/button/i"

        nome_arquivo = driver.find_element(By.XPATH, nome_xpath).text
        nome_limpo = sanitizar_nome(nome_arquivo)
        nome_limpo = nome_limpo[:-4]
        destino_pdf = pasta_destino / f"{nome_limpo}.pdf"

        for tentativa in range(1, MAX_TENTATIVAS + 1):
            try:
                # clique que dispara o PDF (marco de tempo para não confundir com PDF antigo)
                t0 = time.time()
                driver.find_element(By.XPATH, pdf_xpath).click()
                time.sleep(3)
                #py.click(x=2703, y=495)
                py.click(x=784, y=503)
                time.sleep(2)
                #py.click(x=3715, y=146)
                py.click(x=1793, y=146)
                time.sleep(3)
                #py.click(x=2703, y=495)
                py.click(x=784, y=503)
                time.sleep(2)
                #py.click(x=2420, y=20)
                py.click(x=502, y=17)
        
                time.sleep(5)

                pdf_baixado = esperar_download_pdf(DOWNLOAD_DIR, iniciado_em=t0, timeout=15)
                destino_pdf = Path(pasta_destino) / f"{sanitizar_nome(nome_arquivo)}.pdf"

                if pdf_baixado is None:
                    raise TimeoutError(f"Download não concluído na tentativa {tentativa}")

                # sobrescreve se já existir
                if destino_pdf.exists():
                    destino_pdf.unlink()

                shutil.move(str(pdf_baixado), str(destino_pdf))  # move C: -> G: funciona

                print(f"[+] PDF salvo: {destino_pdf}. {now}")
                break  # sucesso — sai do loop de tentativas

            except Exception as e:
                print(f"[!] Tentativa {tentativa}/{MAX_TENTATIVAS} falhou para '{nome_arquivo}': {e}")

                if tentativa < MAX_TENTATIVAS:
                    print(f"    Aguardando {ESPERA_ENTRE_TENTATIVAS}s antes de tentar novamente...")
                    time.sleep(ESPERA_ENTRE_TENTATIVAS)
                else:
                    print(f"[✗] Todas as tentativas falharam para: {nome_arquivo}")
                    raise

    